# 05. Validasi level Bloom (E2)

Ini satu-satunya tahap yang tidak bisa diotomatiskan. Level kognitif hanya ada sebagai **penilaian manusia**,
jadi seseorang harus membaca dan memutuskan. Rubriknya di `protocol/bloom_rubric.md` (C1 sampai C6 untuk
**jawaban**, bukan soal), prosedurnya di `protocol/labeling_sop.md`, dan alat pelabelan luring
`tools/label.html` (klik ganda, tanpa server, tanpa internet, hasilnya CSV).

Urutannya, dan **jangan dibalik**:

1. Kesepakatan antar penilai (kalibrasi dengan 20 item dulu).
2. Baru ukur sistem E2 terhadap konsensus mereka.

**Kenapa kesepakatan diukur lebih dulu.** Itu **langit-langit** yang bisa dicapai mesin. Manusia sepakat 65%
dan mesin 60%: mesin nyaris menyentuh batas atas, melaporkannya sebagai kegagalan justru keliru. Manusia
sepakat 90% dan mesin 60%: kesenjangan nyata. Tanpa angka kesepakatan, akurasi mesin tidak bisa ditafsirkan.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src import config

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 140)
config.ensure_dirs()
GOLD = config.GOLD_DIR / "gold_set.csv"

In [2]:
from src import evaluate_baseline as eb
from src import evaluate_bloom as eblm
from src import viz
from src.agreement import BLOOM_LABELS, compare_raters, score_predictions

LABELS_DIR = config.DATA_DIR / "labels"

## Mengapa kappa berbobot ikut dilaporkan

Level Bloom skala berurutan: kappa biasa memperlakukan ketidaksepakatan C3 vs C4 sama beratnya dengan C1 vs C6.
Dua situasi di bawah punya tingkat kesepakatan **persis sama**; satu selalu meleset satu tingkat, yang lain
tiga tingkat. Laporkan keduanya: yang biasa untuk dibandingkan dengan literatur, yang berbobot untuk gambaran
yang jujur. (Data di sel ini sintetis, hanya untuk memperlihatkan perilaku rumusnya.)

In [3]:
dasar = list("123456") * 3

def geser(langkah, setiap=3):
    hasil = list(dasar)
    for i in range(0, len(dasar), setiap):
        hasil[i] = str(min(6, int(dasar[i]) + langkah))
    return {str(i): v for i, v in enumerate(hasil)}

acuan = {str(i): v for i, v in enumerate(dasar)}
tabel = {}
for nama, langkah in (("selalu meleset 1 tingkat", 1), ("selalu meleset 3 tingkat", 3)):
    r = compare_raters(acuan, geser(langkah))
    tabel[nama] = {"sepakat persis": r.observed, "kappa biasa": r.kappa, "kappa berbobot": r.weighted_kappa}
pd.DataFrame(tabel).T.round(3)

,sepakat persis,kappa biasa,kappa berbobot
selalu meleset 1 tingkat,0.667,0.6,0.939
selalu meleset 3 tingkat,0.667,0.6,0.629


## Tahap 1. Kesepakatan antar penilai

Dua CSV keluaran `tools/label.html` di `data/labels/` (kolom `id`, `label` 1-6 atau `X` untuk tak dapat dinilai).

In [4]:
PENILAI = sorted(LABELS_DIR.glob("penilai*.csv"))[:2] if LABELS_DIR.exists() else []
HAS_LABELS = len(PENILAI) == 2
if not HAS_LABELS:
    print("Belum ada dua CSV penilai di data/labels/ (penilai1.csv, penilai2.csv). Tahap 1 dan 2 dilewati.")
else:
    a, b = (eblm.read_labels(p) for p in PENILAI)
    eblm.print_agreement(a, b, [p.stem for p in PENILAI])
    kesepakatan = compare_raters(a, b)

Belum ada dua CSV penilai di data/labels/ (penilai1.csv, penilai2.csv). Tahap 1 dan 2 dilewati.


In [5]:
if HAS_LABELS and kesepakatan.n:
    viz.plot_confusion(kesepakatan.confusion, BLOOM_LABELS, "Antar penilai", PENILAI[0].stem, PENILAI[1].stem)
    plt.show()

> **Kappa di bawah 0,4: jangan lanjut melabeli.** Baca bersama item yang berbeda, perbaiki rubrik, ulangi
> kalibrasi dengan 20 item baru. Melabeli 300 item pada kappa serendah ini membuang seluruh pekerjaan itu.

## Tahap 2. Ukur sistem E2 terhadap konsensus

Setelah ketidaksepakatan diselesaikan, simpan konsensus sebagai `data/labels/konsensus.csv`, dan jawabannya
(`id`, `text`) di `data/labels/jawaban.csv`. E2 produksi (`engines/bloom.py`) dipanggil langsung.

In [6]:
KONSENSUS, JAWABAN = LABELS_DIR / "konsensus.csv", LABELS_DIR / "jawaban.csv"
if not (HAS_LABELS and KONSENSUS.exists() and JAWABAN.exists()):
    print("konsensus.csv dan jawaban.csv belum ada. Tahap 2 dilewati.")
    HAS_CONSENSUS = False
else:
    eb.bootstrap_backend()
    acuan = eblm.read_labels(KONSENSUS)
    jawaban = eblm.read_answers(JAWABAN)
    prediksi = eblm.predict_with_e2({k: v for k, v in jawaban.items() if k in acuan})
    eblm.print_system(acuan, prediksi, kesepakatan.observed)
    HAS_CONSENSUS = True

konsensus.csv dan jawaban.csv belum ada. Tahap 2 dilewati.


In [7]:
if HAS_CONSENSUS:
    laporan = score_predictions(acuan, prediksi)
    viz.plot_confusion(laporan.confusion, BLOOM_LABELS, "E2 terhadap acuan", "acuan", "sistem")
    plt.show()
    pd.DataFrame(laporan.per_label).T.query("support > 0").round(3)

## Kesimpulan

* Akurasi E2 **belum diukur** (butuh label manusia). Sampai ada angka ini, level Bloom di produk adalah
  taksiran yang belum divalidasi, dan UI harus memperlakukannya demikian.
* Bandingkan akurasi sistem dengan **langit-langit manusia**: selisih ≤ 5 poin berarti sistem praktis
  menyentuh batas atas; selisih lebih besar adalah kesenjangan nyata yang layak diperbaiki.